03 - Forecasting Models

01. Import Libraries
02. Load Train / Test
03. Train-Test Date Check
04. Target & Feature Selection
05. EWM Features
06. Missing Value Check
07. Categorical Features
08. Baseline 1 → Naive
09. Baseline 2 → Moving Average
10. Baseline 3 → EWM
11. Evaluation Metrics
12. Time-Based Validation
13. XGBoost
14. LightGBM
15. XGBoost vs LightGBM
16. Log Target Experiment
17. Hyperparameter Tuning
18. Feature Importance
19. Error Analysis
20. Final Model Selection
21. Final Test Evaluation
22. Future Demand Forecast
23. Forecast Output

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:

#Load Train / test
train = pd.read_csv("train-2.csv")
test = pd.read_csv("test-2.csv")

train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain date:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date:")
print(test["Date"].min(), "->", test["Date"].max())

In [ ]:
#Train-Test Date Check / Sort

train = train.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

test = test.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

In [ ]:
# Target & Feature Selection

target = "Units Sold"

y_train = train[target]
y_test = test[target]

In [ ]:
print(y_train.describe())

In [ ]:
print(train.columns.tolist())

In [ ]:
# EWM Features
#EWM 7 & 30

all_data = pd.concat(
    [train, test],
    ignore_index=True
)

all_data = all_data.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

all_data["sales_ewm_7"] = (
    all_data
    .groupby(["Store ID", "Product ID"])["Units Sold"]
    .transform(
        lambda x: x.shift(1).ewm(
            span=7,
            adjust=False
        ).mean()
    )
)

all_data["sales_ewm_30"] = (
    all_data
    .groupby(["Store ID", "Product ID"])["Units Sold"]
    .transform(
        lambda x: x.shift(1).ewm(
            span=30,
            adjust=False
        ).mean()
    )
)

train_end_date = train["Date"].max()

train = all_data[
    all_data["Date"] <= train_end_date
].copy()

test = all_data[
    all_data["Date"] > train_end_date
].copy()

In [ ]:
# EWM kontrol
train[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Units Sold",
        "sales_ewm_7",
        "sales_ewm_30"
    ]
].head(15)

In [ ]:
drop_cols = [
    "Units Sold",
    "Date",
    "Units Ordered"  # Sızıntıyı (leakage) engellemek için eklendi
]

for maybe_leak in ["Demand Forecast", "Demand_Forecast_Baseline",
                    "Store ID_Code", "Product ID_Code", "Category_Code"]:
    if maybe_leak in train.columns:
        drop_cols.append(maybe_leak)

print("drop_cols:", drop_cols)

In [ ]:
X_train = train.drop(columns=drop_cols)
y_train = train["Units Sold"]

X_test = test.drop(columns=drop_cols)
y_test = test["Units Sold"]

print(X_train.dtypes)

In [ ]:

categorical_cols = [
    c for c in X_train.columns
    if pd.api.types.is_string_dtype(X_train[c])
    or isinstance(X_train[c].dtype, pd.CategoricalDtype)
]
print("categorical_cols:", categorical_cols)

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

X_train.dtypes.value_counts()

In [ ]:
# Baseline modelleri train verisi üzerinde hesaplama
val_baseline = train.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

# 1. Naive (Önceki günün satışı)
val_baseline["naive_prediction"] = (
    val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"].shift(1)
)

# 2. 7 Günlük Hareketli Ortalama
val_baseline["ma_7_prediction"] = (
    val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# 3. 7 Günlük Üstel Ağırlıklı Ortalama (EWM)
val_baseline["ewm_7_prediction"] = (
    val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"]
    .transform(lambda x: x.shift(1).ewm(span=7, adjust=False).mean())
)

# Yalnızca validation dönemini (son 30 günü) filtreleme
# Not: validation_start değişkeni 16. hücrede tanımlanıyor; 
# bu hücre 16. hücreden sonra çalıştırılmalı veya validation_start burada hesaplanmalı:
val_start_date = train["Date"].max() - pd.Timedelta(days=30 - 1)
val_baseline_eval = val_baseline[val_baseline["Date"] >= val_start_date].copy()

print("Validation Baseline Shape:", val_baseline_eval.shape)
val_baseline_eval[["naive_prediction", "ma_7_prediction", "ewm_7_prediction"]].isna().mean()

In [ ]:
def mae(y_true, y_pred):
    return mean_absolute_error(
        y_true,
        y_pred
    )

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def smape(y_true, y_pred):
    denominator = (
        np.abs(y_true) +
        np.abs(y_pred)
    )
    return np.mean(
        2 * np.abs(y_true - y_pred) /
        np.where(denominator == 0, 1, denominator)
    ) * 100

def wape(y_true, y_pred):
    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    ) * 100

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
        "WAPE": wape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

In [ ]:
# Naive Evaluation (Validation üzerinde)
naive_eval = val_baseline_eval.dropna(subset=["naive_prediction"])
naive_results = evaluate_model(
    naive_eval["Units Sold"],
    naive_eval["naive_prediction"]
)

# Moving Average 7 Evaluation (Validation üzerinde)
ma_eval = val_baseline_eval.dropna(subset=["ma_7_prediction"])
ma_results = evaluate_model(
    ma_eval["Units Sold"],
    ma_eval["ma_7_prediction"]
)

# EWM 7 Evaluation (Validation üzerinde)
ewm_eval = val_baseline_eval.dropna(subset=["ewm_7_prediction"])
ewm_results = evaluate_model(
    ewm_eval["Units Sold"],
    ewm_eval["ewm_7_prediction"]
)

baseline_results = pd.DataFrame({
    "Naive": naive_results,
    "Moving Average 7": ma_results,
    "EWM 7": ewm_results
}).T

baseline_results

In [ ]:
validation_days = 30

validation_start = (
    train["Date"].max()
    - pd.Timedelta(days=validation_days - 1)
)

train_model = train[
    train["Date"] < validation_start
].copy()

validation = train[
    train["Date"] >= validation_start
].copy()

print("Model train:")
print(train_model["Date"].min(), "->", train_model["Date"].max())

print("\nValidation:")
print(validation["Date"].min(), "->", validation["Date"].max())

In [ ]:
X_tr = train_model.drop(
    columns=drop_cols
)
y_tr = train_model[target]

X_val = validation.drop(
    columns=drop_cols
)
y_val = validation[target]

for col in categorical_cols:
    X_tr[col] = X_tr[col].astype("category")
    X_val[col] = X_val[col].astype("category")

In [ ]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    enable_categorical=True,
    early_stopping_rounds=50  
)

In [ ]:
xgb_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

In [ ]:
xgb_val_pred = xgb_model.predict(X_val)

In [ ]:
xgb_val_results = evaluate_model(
    y_val,
    xgb_val_pred
)

xgb_val_results

In [ ]:
from lightgbm import early_stopping, log_evaluation

lgbm_model = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [ ]:
from lightgbm import early_stopping

lgbm_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

In [ ]:
lgbm_val_pred = lgbm_model.predict(
    X_val
)

In [ ]:
lgbm_val_results = evaluate_model(
    y_val,
    lgbm_val_pred
)

lgbm_val_results

In [ ]:
validation_results = pd.DataFrame({
    "XGBoost": xgb_val_results,
    "LightGBM": lgbm_val_results
}).T

validation_results

In [ ]:
y_tr_log = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

In [ ]:
xgb_log_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    enable_categorical=True
)

In [ ]:
xgb_log_model.fit(
    X_tr,
    y_tr_log,
    eval_set=[
        (X_val, y_val_log)
    ],
    verbose=False
)

In [ ]:
xgb_log_pred = xgb_log_model.predict(
    X_val
)

xgb_log_pred = np.expm1(
    xgb_log_pred
)

In [ ]:
xgb_log_results = evaluate_model(
    y_val,
    xgb_log_pred
)

xgb_log_results

In [ ]:
lgbm_log_model = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

In [ ]:
lgbm_log_model.fit(
    X_tr,
    y_tr_log,
    categorical_feature=categorical_cols
)

In [ ]:
lgbm_log_pred = lgbm_log_model.predict(
    X_val
)

lgbm_log_pred = np.expm1(
    lgbm_log_pred
)

In [ ]:
lgbm_log_results = evaluate_model(
    y_val,
    lgbm_log_pred
)

lgbm_log_results

In [ ]:
all_validation_results = pd.DataFrame({
    "XGBoost": xgb_val_results,
    "XGBoost + Log": xgb_log_results,
    "LightGBM": lgbm_val_results,
    "LightGBM + Log": lgbm_log_results
}).T

all_validation_results

In [ ]:
final_comparison = pd.concat([
    baseline_results,
    all_validation_results
])

final_comparison

In [ ]:
xgb_importance = pd.DataFrame({
    "Feature": X_tr.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

xgb_importance.head(20)

In [ ]:
lgbm_importance = pd.DataFrame({
    "Feature": X_tr.columns,
    "Importance": lgbm_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

lgbm_importance.head(20)

In [ ]:
import seaborn as sns

def plot_comparative_importance(xgb_mod, lgbm_mod, feature_names, top_n=15):
    xgb_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": xgb_mod.feature_importances_
    }).sort_values(by="Importance", ascending=False).head(top_n)

    lgbm_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": lgbm_mod.feature_importances_
    }).sort_values(by="Importance", ascending=False).head(top_n)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False)
    sns.set_theme(style="whitegrid")

    # XGBoost
    sns.barplot(
        x="Importance", 
        y="Feature", 
        hue="Feature", 
        data=xgb_df, 
        ax=axes[0], 
        palette="Blues_r", 
        legend=False
    )
    axes[0].set_title(f"Top {top_n} Feature Importance - XGBoost", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("Önem Düzeyi (Weight / Gain)", fontsize=12)
    axes[0].set_ylabel("Öznitelikler", fontsize=12)

    # LightGBM
    sns.barplot(
        x="Importance", 
        y="Feature", 
        hue="Feature", 
        data=lgbm_df, 
        ax=axes[1], 
        palette="Greens_r", 
        legend=False
    )
    axes[1].set_title(f"Top {top_n} Feature Importance - LightGBM", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Bölünme Sayısı (Split Count)", fontsize=12)
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()

# Fonksiyonu çağırma
plot_comparative_importance(xgb_model, lgbm_model, X_tr.columns, top_n=15)

In [ ]:
# Hata / Kalıntı Analizi (En iyi model olan LightGBM üzerinden)
residuals = y_val - lgbm_val_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Kalıntıların Dağılımı (Histogram)
axes[0].hist(residuals, bins=50, color="#1f77b4", edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_title("Kalıntı Dağılımı (Residuals: Actual - Predicted)")
axes[0].set_xlabel("Tahmin Hatası (Hata = Gerçek - Tahmin)")
axes[0].set_ylabel("Frekans")

# 2. Gerçek Değer vs. Tahmin Edilen Değer (Scatter)
axes[1].scatter(y_val, lgbm_val_pred, alpha=0.3, color="#2ca02c", s=15)
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], color="red", linestyle="--", lw=1.5)
axes[1].set_title("Gerçek Değer vs. Tahmin")
axes[1].set_xlabel("Gerçek Satış (Units Sold)")
axes[1].set_ylabel("Tahmin Edilen Satış (Predicted)")

plt.tight_layout()
plt.show()

In [ ]:
#Hyperparameter Tuning

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# 1. Zaman serisi cross-validation ayrımı
tscv = TimeSeriesSplit(n_splits=3)

# 2. Aranacak parametre ızgarası
lgbm_param_grid = {
    "n_estimators": [300, 500, 800],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [15, 31, 63],
    "colsample_bytree": [0.7, 0.8],
    "subsample": [0.8],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [0.0, 1.0]
}

base_lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1
)

# 3. GridSearchCV
lgbm_grid_search = GridSearchCV(
    estimator=base_lgbm,
    param_grid=lgbm_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

lgbm_grid_search.fit(X_tr, y_tr)

print("En İyi Parametreler:", lgbm_grid_search.best_params_)
print("En İyi CV MAE Skoru:", -lgbm_grid_search.best_score_)

# 4. En iyi modelle validation değerlendirmesi
best_lgbm_model = lgbm_grid_search.best_estimator_
best_lgbm_val_pred = best_lgbm_model.predict(X_val)
best_lgbm_val_results = evaluate_model(y_val, best_lgbm_val_pred)

print("\nTuned LightGBM Validation Sonuçları:")
print(best_lgbm_val_results)